Localization is the [Bayes filter](/aiml-common/lectures/rse/recursive-state-estimation/index) applied to the pose of the agent itself. The agent is given a map $m$ of its environment and keeps a belief over its pose $s$, which it updates from its actions $a$ and its measurements $z$. This section first adds the map to the Bayes filter, then tracks the pose of a ground robot with the extended Kalman filter.

## Localization with a known map

![Graphical model for localization: each state depends on the previous state, the current action and the map, and each measurement depends on the current state, its correspondence and the map.](images/pgm-localization.svg)
*Graphical model for localization of a mobile agent with known correspondences. Shaded nodes are observed: the actions $a$, the measurements $z$, the correspondences $c$ and the known map $m$. The dashed edges from the map to the states belong to a map-based motion model, which the EKF below leaves out.*

For localization the agent is also given a map $m$. Maps take several forms. An indoor robot can use a floor plan or an [occupancy grid](/aiml-common/lectures/rse/occupancy-mapping/index). The EKF in this section uses a *landmark map*, a list of point features with known positions. A car uses a lane-level road map that records the lane markings, traffic signs and other landmarks along each road. The [last part of this section](#where-road-maps-come-from) describes how such maps are built.

A known map does not make the pose unique. The figure below shows a corridor of offices whose layout repeats every 6 m. The robot faces along the corridor and takes one range scan all around it. The belief over its position, with the heading known, is the likelihood of that scan at every point of the corridor.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

plt.rcParams.update(
    {
        "figure.figsize": (6.5, 3.5),  # matches the 6.5 in text column
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.prop_cycle": cycler(
            color=["#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#1d99e0", "#7f7f7f"]
        ),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "svg.fonttype": "path",
    }
)

# named colors from the palette above
BLUE, VERMILLION, GREEN, PINK, GOLD, SKY, GRAY = (
    "#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#1d99e0", "#7f7f7f",
)

np.random.seed(7)

In [ ]:
# A corridor of offices whose layout repeats every 6 m: wall segments (x0, y0, x1, y1) in meters.
walls = [(0, 0, 0, 2), (14, 0, 14, 2)]  # end walls
for x0 in (0, 6):
    walls += [(x0 + 2, 2, x0 + 2, 4), (x0 + 5, 2, x0 + 5, 4), (x0 + 2, 4, x0 + 5, 4)]  # office above, door at x0 + 3
    walls += [(x0 + 4, 0, x0 + 4, -2), (x0 + 7, 0, x0 + 7, -2), (x0 + 4, -2, x0 + 7, -2)]  # office below, door at x0 + 5
top = [0, 3, 4, 9, 10, 14]  # corridor wall y = 2, with door gaps [3, 4] and [9, 10]
bottom = [0, 5, 6, 11, 12, 14]  # corridor wall y = 0, with door gaps [5, 6] and [11, 12]
walls += [(top[k], 2, top[k + 1], 2) for k in (0, 2, 4)] + [(bottom[k], 0, bottom[k + 1], 0) for k in (0, 2, 4)]
walls = np.array(walls, dtype=float)

scan_range, n_beams, sigma_scan = 2.5, 12, 0.25
beam = np.linspace(-np.pi, np.pi, n_beams, endpoint=False)  # heading known: the robot faces +x


def scan(p):
    """Ranges of all beams from the positions p (shape (n, 2)), up to scan_range."""
    u = np.stack([np.cos(beam), np.sin(beam)], axis=1)  # (beams, 2)
    a, e = walls[:, :2], walls[:, 2:] - walls[:, :2]  # segment start and direction
    w = a[None, None] - p[:, None, None]  # (n, 1, segments, 2)
    den = u[None, :, None, 0] * e[None, None, :, 1] - u[None, :, None, 1] * e[None, None, :, 0]
    with np.errstate(divide="ignore", invalid="ignore"):
        t = (w[..., 0] * e[None, None, :, 1] - w[..., 1] * e[None, None, :, 0]) / den
        k = (w[..., 0] * u[None, :, None, 1] - w[..., 1] * u[None, :, None, 0]) / den
    t = np.where((t > 0) & (k >= 0) & (k <= 1), t, np.inf)
    return np.minimum(t.min(axis=2), scan_range)


s_ambig = np.array([3.5, 1.0])
z_ambig = scan(s_ambig[None])[0] + np.random.default_rng(1).normal(0, sigma_scan, n_beams)
gx, gy = np.meshgrid(np.arange(0.25, 13.76, 0.05), np.arange(0.15, 1.86, 0.05))
loglik = -((scan(np.column_stack([gx.ravel(), gy.ravel()])) - z_ambig) ** 2).sum(axis=1) / (2 * sigma_scan**2)
belief_ambig = np.exp(loglik - loglik.max()).reshape(gx.shape)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.pcolormesh(gx, gy, belief_ambig**0.25, cmap="Blues", shading="auto", rasterized=True)  # power for visibility
for x0, y0, x1, y1 in walls:
    ax.plot([x0, x1], [y0, y1], color="black", lw=1.5)
ax.plot(*s_ambig, "s", ms=9, mfc="none", mec=GREEN, mew=1.5, label="true position")
ax.annotate("", xy=s_ambig + [0.8, 0], xytext=s_ambig + [0.2, 0], arrowprops={"arrowstyle": "->", "color": GREEN, "lw": 1.5})
peak = np.column_stack([gx.ravel(), gy.ravel()])[np.argmax(np.where(gx.ravel() > 7, belief_ambig.ravel(), 0))]
ax.plot(*peak, "o", ms=14, mfc="none", mec=VERMILLION, mew=1.5, label="second peak of the belief")
ax.set_aspect("equal")
ax.set_xlim(-0.3, 14.3)
ax.set_ylim(-2.3, 4.3)
ax.set_xlabel("$x$ (m)")
ax.set_ylabel("$y$ (m)")
ax.grid(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
fig.tight_layout()
plt.show()

*Belief over the robot's position after one range scan in a corridor whose layout repeats every 6 m (darker is more probable). The scan from the true position (square) matches equally well 6 m further along, so the belief has two peaks.*

The Bayes filter for localization conditions on the map. One step of the filter takes the previous belief $\mathtt{bel}(s_{t-1})$, the action $a_t$, the measurement $z_t$ and the map $m$, and returns the new belief $\mathtt{bel}(s_t)$:

$$
\begin{array}{lll}
1: & \textbf{for all } s_t \textbf{ do} & \\
2: & \qquad \widehat{\mathtt{bel}}(s_t) = \int p(s_t \mid a_t, s_{t-1}, m)\, \mathtt{bel}(s_{t-1})\, ds_{t-1} & \text{prediction} \\
3: & \qquad \mathtt{bel}(s_t) = \eta\, p(z_t \mid s_t, m)\, \widehat{\mathtt{bel}}(s_t) & \text{measurement update} \\
4: & \textbf{end for} & \\
5: & \textbf{return } \mathtt{bel}(s_t) &
\end{array}
$$

This is Markov localization, Table 7.1 of *Probabilistic Robotics*. The map conditions both steps. In the measurement update it says what the robot should observe from each pose. In the prediction it can rule poses out: the robot cannot end up inside a wall. Section 5.5 of the book approximates this *map-based motion model* for short steps by multiplying the map-free model by the consistency of the pose with the map,

$$
p(s_t \mid a_t, s_{t-1}, m) = \eta\, p(s_t \mid a_t, s_{t-1})\, p(s_t \mid m),
$$

where $p(s_t \mid m)$ is zero for poses that collide with an obstacle and constant elsewhere. The EKF below uses the map-free velocity model $g$, which amounts to assuming $p(s_t \mid a_t, s_{t-1}, m) = p(s_t \mid a_t, s_{t-1})$. That is reasonable for a robot driving in open space among point landmarks. A Gaussian could not represent a belief cut off by a wall anyway.

## EKF localization with known correspondences

EKF localization tracks the pose of a ground robot such as the TurtleBot, $s_t = (x, y, \theta)$, with the [extended Kalman filter](/aiml-common/lectures/rse/ekf/index). Its belief is a single Gaussian, which has one peak, so the filter has to start from a rough estimate of where the robot is. It cannot express a belief like "the robot is in one of these two places", which is what you get when the starting pose is unknown. The [particle filter](/aiml-common/lectures/rse/particle-filter/index) can represent such a belief.

### Landmarks and features

The filter does not use raw scans or images. A *feature extractor* reduces each scan or image to a short list of *features*, one for each detected landmark, and the measurement model explains those features. Inference over a few features can be orders of magnitude cheaper than inference over the full measurement (*Probabilistic Robotics*, Section 6.6). The [beam model](/aiml-common/lectures/sensor-models/beam/index), in contrast, explains every reading of a scan. An EKF cannot use the beam model directly. Its likelihood is a mixture with a point mass at the maximum range, not a Gaussian, and its expected range comes from ray casting, which has no useful derivative with respect to the pose: a small turn can move a ray from a near object to a far wall. A [particle filter](/aiml-common/lectures/rse/particle-filter/index) only needs to evaluate the likelihood at each particle's pose, so scan-based localization usually runs on one, as Nav2's AMCL does.

A *landmark* is a distinct object that the robot uses for navigation, such as a pole, a door post or a tree trunk. In the most common landmark model, the sensor measures the landmark's range $r$ and bearing $\phi$ in the robot's frame. The extractor may also report a *signature*, which describes how the landmark looks, such as its color, size or type. Thrun, Burgard and Fox write the signature as $s$. In this course $s_t$ is the robot's state, so the signature is written as $\kappa$.

The robot's map $m$ is the list of landmarks it knows, $m = \{m_1, \dots, m_N\}$. Landmark $j$ has a position $(m_{j,x}, m_{j,y})$ in the map frame and a signature $\kappa_j$. This is a *feature-based map*: it records where the landmarks are and what they look like, but nothing about the space between them. An [occupancy grid](/aiml-common/lectures/rse/occupancy-mapping/index), in contrast, records whether every cell of the space is free or occupied.

At each step $t$ the robot sees some of the landmarks and extracts one feature from each. The index $i$ counts these features within the step, so $f^i_t = (r^i_t, \phi^i_t, \kappa^i_t)$ is the $i$-th feature at time $t$, and the features of a measurement $z_t$ form the set $f(z_t) = \{f^1_t, f^2_t, \dots\}$. The index $j$ counts the landmarks in the map. The two differ because the robot sees only some of the landmarks, in no fixed order, so its first feature at a step may come from landmark 3. The number of features can change from step to step. The model treats the noise in each feature as independent of the noise in every other feature, so the likelihood factors over the features and a filter can process them one at a time:

$$
p(f(z_t) \mid s_t, m) = \prod_i p(r^i_t, \phi^i_t, \kappa^i_t \mid s_t, m).
$$

The *correspondence variable* $c^i_t = j$ says that feature $i$ at time $t$ came from landmark $j$. Thrun, Burgard and Fox also allow $c^i_t = N + 1$ for a feature that matches no landmark in the map, such as a spurious detection or a landmark that the map does not yet contain. With the robot at $s_t = (x, y, \theta)$ and $c^i_t = j$, the feature equals the noise-free geometry plus independent Gaussian noise:

$$
\begin{bmatrix} r^i_t \\ \phi^i_t \\ \kappa^i_t \end{bmatrix} =
\begin{bmatrix} \sqrt{(m_{j,x} - x)^2 + (m_{j,y} - y)^2} \\ \operatorname{atan2}(m_{j,y} - y,\; m_{j,x} - x) - \theta \\ \kappa_j \end{bmatrix} +
\begin{bmatrix} \varepsilon_r \\ \varepsilon_\phi \\ \varepsilon_\kappa \end{bmatrix},
$$

where $\varepsilon_r$, $\varepsilon_\phi$ and $\varepsilon_\kappa$ have zero mean and variances $\sigma_r^2$, $\sigma_\phi^2$ and $\sigma_\kappa^2$. When the correspondence is known, the likelihood of a feature is the product of three Gaussian densities, one for each error (*Probabilistic Robotics*, Table 6.4):

$$
p(f^i_t \mid c^i_t = j, s_t, m) = \mathcal{N}(r^i_t - \hat r;\, 0, \sigma_r^2)\; \mathcal{N}(\phi^i_t - \hat\phi;\, 0, \sigma_\phi^2)\; \mathcal{N}(\kappa^i_t - \kappa_j;\, 0, \sigma_\kappa^2),
$$

where $\hat r$ and $\hat\phi$ are the range and bearing that landmark $j$ would produce from pose $s_t$, and the bearing error is wrapped to $[-\pi, \pi)$.

The filter below is the same for every sensor. What changes is the measurement noise $R_t$, which landmarks count as visible, and whether the signature determines the correspondence. The [Perception](/book/perception/index#what-each-sensor-measures) overview lists what a LIDAR, a camera and an RGB-D camera each supply. A camera that detects [fiducial markers](https://en.wikipedia.org/wiki/Fiducial_marker) reads each marker's ID, which names the landmark, so the robot reads $c^i_t$ from the signature. A 2D LIDAR such as the TurtleBot3's LDS-01 returns only ranges. Landmarks extracted from its scan, such as poles or corners, carry no ID, so the robot has to work out which landmark produced each feature. That is [correspondence estimation](/aiml-common/lectures/rse/correspondence-estimation/index). This section assumes known correspondences, as with fiducial markers.

The examples below use a range noise of $\sigma_r = 5$ cm, a bearing noise of $\sigma_\phi = 2°$ and a sensing range of 1.5 m. These are illustrative values, not a calibration. They are of the same order as the LDS-01 at short range, whose specified accuracy beyond 0.5 m is ±5% ([ROBOTIS](https://emanual.robotis.com/docs/en/platform/turtlebot3/appendix_lds_01/)), about 6 cm at 1.25 m. A specified accuracy bounds the error, while $\sigma_r$ is a standard deviation. For a real sensor you would estimate both standard deviations from calibration data, by measuring landmarks at known positions.

The landmark positions are treated as exact. A surveyed map has errors of its own, and a filter that ignores them is overconfident. If the errors of different landmarks are independent, each landmark's covariance can be propagated through $h$ and added to the measurement noise $R_t$. When the landmark positions are estimated together with the pose, the problem becomes [SLAM](/aiml-common/lectures/rse/slam/index).

### Motion model

The action is the pair of wheel speeds $a_t = (u_L, u_R)$. As [derived in nonholonomic robots](/aiml-common/lectures/kinematics/nonholonomic-robots/index#wheel-speeds-of-a-differential-drive), they set the forward speed $v = \frac{r}{2}(u_L + u_R)$ and the turning rate $\omega = \frac{r}{2d}(u_R - u_L)$. Holding $v$ and $\omega$ constant for $\Delta t$ is a constant body twist $\mathcal{V}_b = (\omega, v, 0)$. The new pose is then the planar screw motion of [motion representations](/aiml-common/lectures/motion-algebras/motion-representations/index#from-velocity-to-motion), $T_t = T_{t-1}\exp([\mathcal{V}_b]\,\Delta t)$, which moves the robot along an arc of radius $v/\omega$. Multiplying out the two matrices and reading off the position and the heading gives the motion model

$$
g(a_t, s_{t-1}) = \begin{bmatrix} x - \frac{v}{\omega}\sin\theta + \frac{v}{\omega}\sin(\theta + \omega\Delta t) \\ y + \frac{v}{\omega}\cos\theta - \frac{v}{\omega}\cos(\theta + \omega\Delta t) \\ \theta + \omega\Delta t \end{bmatrix}.
$$

Its Jacobian with respect to the state is

$$
G_t = \begin{bmatrix} 1 & 0 & \frac{v}{\omega}\left(\cos(\theta + \omega\Delta t) - \cos\theta\right) \\ 0 & 1 & \frac{v}{\omega}\left(\sin(\theta + \omega\Delta t) - \sin\theta\right) \\ 0 & 0 & 1 \end{bmatrix},
$$

evaluated at $\mu_{t-1}$. The noise comes from the wheels. Each wheel turns at its commanded speed plus Gaussian noise that stays constant during the step. $M_t$ is the $2 \times 2$ covariance of that noise, in wheel-speed units of $(\text{rad/s})^2$. The Jacobian $V_t = \partial g / \partial a_t$ maps that noise into the state, so the motion noise covariance is $Q_t = V_t M_t V_t^\top$.

#### Deriving the noise Jacobian $V_t$

The wheel speeds enter $g$ only through $v$ and $\omega$, so the chain rule splits $V_t$ into two factors:

$$
V_t = \frac{\partial g}{\partial a_t} = \frac{\partial g}{\partial (v, \omega)} \, \frac{\partial (v, \omega)}{\partial (u_L, u_R)} .
$$

The second factor is a constant matrix, because $v$ and $\omega$ are linear in the wheel speeds:

$$
J = \frac{\partial (v, \omega)}{\partial (u_L, u_R)} = \begin{bmatrix} \dfrac{r}{2} & \dfrac{r}{2} \\[6pt] -\dfrac{r}{2d} & \dfrac{r}{2d} \end{bmatrix}.
$$

Each column of $J$ belongs to one wheel. An error in one wheel's speed changes $v$ by $r/2$ times that error and $\omega$ by $\mp r/(2d)$ times it, so noise on a single wheel both moves the robot forward and turns it.

The first factor is the derivative of the arc. Write $\theta' = \theta + \omega\Delta t$ for the heading at the end of the step. Differentiating $g$ with respect to $v$ gives

$$
\frac{\partial g}{\partial v} = \frac{1}{\omega} \begin{bmatrix} \sin\theta' - \sin\theta \\ \cos\theta - \cos\theta' \\ 0 \end{bmatrix},
$$

and with respect to $\omega$, where $\theta'$ also depends on $\omega$,

$$
\begin{aligned}
\frac{\partial g_x}{\partial \omega} &= \frac{v}{\omega^2}\left(\sin\theta - \sin\theta'\right) + \frac{v\,\Delta t}{\omega}\cos\theta', \\
\frac{\partial g_y}{\partial \omega} &= \frac{v}{\omega^2}\left(\cos\theta' - \cos\theta\right) + \frac{v\,\Delta t}{\omega}\sin\theta', \\
\frac{\partial g_\theta}{\partial \omega} &= \Delta t .
\end{aligned}
$$

The two columns form the $3 \times 2$ matrix $\partial g / \partial (v, \omega)$, and

$$
V_t = \begin{bmatrix} \dfrac{\partial g}{\partial v} & \dfrac{\partial g}{\partial \omega} \end{bmatrix} J ,
$$

a $3 \times 2$ matrix evaluated at $\mu_{t-1}$ and the commanded wheel speeds. With $M_t$ of size $2 \times 2$, the motion noise $Q_t = V_t M_t V_t^\top$ is $3 \times 3$, the size of the state covariance. $V_t$ already contains $\Delta t$, so it turns the wheel-speed covariance into a pose covariance directly, and $Q_t$ needs no extra factor of $\Delta t$. For the same reason $M_t$ must describe wheel speeds, not wheel angles.

When both wheels turn at the same speed, $\omega = 0$ and the formulas above divide by zero. Their limit as $\omega \to 0$ is the straight drive:

$$
\frac{\partial g}{\partial v} = \Delta t \begin{bmatrix} \cos\theta \\ \sin\theta \\ 0 \end{bmatrix}, \qquad
\frac{\partial g}{\partial \omega} = \begin{bmatrix} -\frac{1}{2} v\,\Delta t^2 \sin\theta \\ \frac{1}{2} v\,\Delta t^2 \cos\theta \\ \Delta t \end{bmatrix}.
$$

A turn-rate error during a straight drive moves the robot sideways by an amount that grows with $\Delta t^2$, because the heading error it causes grows during the step. The code uses these limits when the turn per step is tiny, $|\omega\Delta t| < 10^{-6}$. Near $\omega = 0$ the arc formulas subtract nearly equal numbers and divide by a small $\omega$, which loses precision. The function `V` in the code below implements both cases.

### Measurement model

For landmark $j$, write $\delta = (m_{j,x} - x,\; m_{j,y} - y)$ and $q = \delta^\top\delta$. The predicted measurement and its Jacobian are

$$
h(s_t, j, m) = \begin{bmatrix} \sqrt{q} \\ \operatorname{atan2}(\delta_y, \delta_x) - \theta \end{bmatrix}, \qquad H^i_t = \frac{1}{q}\begin{bmatrix} -\sqrt{q}\,\delta_x & -\sqrt{q}\,\delta_y & 0 \\ \delta_y & -\delta_x & -q \end{bmatrix},
$$

evaluated at the predicted mean $\bar\mu_t$, with measurement noise covariance $R_t = \operatorname{diag}(\sigma_r^2, \sigma_\phi^2)$. The signature is left out of $h$. It does not depend on the robot's pose, so once the correspondence is known it carries no information for the update (Section 7.5 of *Probabilistic Robotics*). Both $h$ and $H^i_t$ are undefined when the robot stands on the landmark, $q = 0$, and the bearing row of $H^i_t$ grows as $1/\sqrt q$ close to it. A real system rejects features with a very small predicted range.

With the landmark $j$ and the map $m$ fixed, $h$ maps the 3-dimensional pose to a 2-dimensional feature, $\mathbb R^3 \to \mathbb R^2$, and $H^i_t$ is $2 \times 3$, as in the size table of the [EKF section](/aiml-common/lectures/rse/ekf/index). Its zero entry says that the range does not depend on the heading, and its $-1$ says that the bearing falls by one radian for each radian the robot turns. $R_t$ is $2 \times 2$, and so is the innovation covariance $S^i_t = H^i_t \bar\Sigma_t H^{i\top}_t + R_t$ of line 5 below, which makes the gain $K^i_t$ a $3 \times 2$ matrix. The algorithm processes the features of a step one at a time, so every update has these sizes. Stacking all $K$ features of a step into one measurement would instead give $h: \mathbb R^3 \to \mathbb R^{2K}$ and a $2K \times 3$ Jacobian.

### The algorithm

One step takes the previous belief, the action, the measurements and their correspondences, and the map:

$$
\begin{array}{ll}
1: & \bar\mu_t = g(a_t, \mu_{t-1}) \\
2: & \bar\Sigma_t = G_t\, \Sigma_{t-1}\, G_t^\top + V_t M_t V_t^\top \\
3: & \text{for each observed feature } z^i_t \text{ with } j = c^i_t: \\
4: & \qquad \hat z^i_t = h(\bar\mu_t, j, m), \quad \text{and } H^i_t \text{ at } \bar\mu_t \\
5: & \qquad S^i_t = H^i_t \bar\Sigma_t H^{i\top}_t + R_t \\
6: & \qquad \nu^i_t = z^i_t - \hat z^i_t, \quad \text{with its bearing wrapped to } [-\pi, \pi) \\
7: & \qquad \text{if } \nu^{i\top}_t (S^i_t)^{-1} \nu^i_t > \gamma: \text{ skip this feature} \\
8: & \qquad K^i_t = \bar\Sigma_t H^{i\top}_t (S^i_t)^{-1} \\
9: & \qquad \bar\mu_t \leftarrow \bar\mu_t + K^i_t \nu^i_t \\
10: & \qquad \bar\Sigma_t \leftarrow \left(I - K^i_t H^i_t\right) \bar\Sigma_t \left(I - K^i_t H^i_t\right)^\top + K^i_t R_t K^{i\top}_t \\
11: & \mu_t = \bar\mu_t, \quad \Sigma_t = \bar\Sigma_t
\end{array}
$$

Line 6 wraps the bearing of the innovation $\nu^i_t$, since an angle error of $2\pi - 0.01$ is really $-0.01$. The heading in the state is wrapped the same way after each update. The filter treats $(x, y, \theta)$ as ordinary coordinates near its mean and only has to take care where the heading crosses $\pm\pi$.

Line 7 is a *validation gate*. The squared Mahalanobis distance $\nu^\top S^{-1} \nu$ measures how far a feature is from its prediction, in units of the predicted spread. If the models are right it follows a chi-square distribution with 2 degrees of freedom and exceeds $\gamma = 9.21$ only 1% of the time. A known ID does not make a feature valid. A misdetection, a partly hidden marker or a reflection can produce a feature far from its prediction, and one such feature can pull the estimate away. The gate drops it. The examples below run with the gate off, and the last of them shows what the gate does.

Line 10 is the Joseph form of the covariance update. For the optimal gain it equals $(I - K^i_t H^i_t)\bar\Sigma_t$. It is written as a sum of two positive semidefinite terms, so rounding errors over many updates cannot make the covariance lose its symmetry or become indefinite.

The loop uses one feature at a time. Each update starts from the belief left by the previous one. For a linear-Gaussian model with independent measurement noises this gives the same result as one update with all features stacked. The EKF relinearizes $h$ at the updated mean before each feature, so its result can depend slightly on the order of the features and differ from a stacked update.

This is the EKF localization algorithm with known correspondences, Table 7.2 of *Probabilistic Robotics* (Thrun, Burgard and Fox), with $s$ for the state, wheel speeds for the action, $Q$ and $R$ as named in this course, and the gate and the Joseph form added. With unknown correspondences (Table 7.3), the robot computes $\hat z$ and $S$ for every landmark, assigns each feature to the landmark under which it is most likely, and rejects features that no landmark explains. That is the subject of [correspondence estimation](/aiml-common/lectures/rse/correspondence-estimation/index).

In [ ]:
r, d = 0.033, 0.1435  # TurtleBot3 Waffle wheel radius and half wheel separation, in meters
dt = 0.5  # seconds per step
STRAIGHT = 1e-6  # below this turn angle |w dt| per step, use the straight-line limits


def wrap(angle):
    """Wrap an angle to [-pi, pi)."""
    return (angle + np.pi) % (2 * np.pi) - np.pi


def twist(a):
    """Forward speed v and turning rate w from the wheel speeds a = (u_L, u_R)."""
    u_L, u_R = a
    return r / 2 * (u_L + u_R), r / (2 * d) * (u_R - u_L)


J = np.array([[r / 2, r / 2], [-r / (2 * d), r / (2 * d)]])  # d(v, w) / d(u_L, u_R)


def g(a, s, dt=dt):
    """Velocity motion model: the pose after holding the wheel speeds a for dt.

    s is one pose (x, y, theta) or an array of shape (3, n) of poses.
    """
    v, w = twist(a)
    x, y, th = s
    if abs(w * dt) < STRAIGHT:  # straight line, the limit of the arc
        return np.array([x + v * dt * np.cos(th), y + v * dt * np.sin(th), wrap(th)])
    return np.array([
        x - v / w * np.sin(th) + v / w * np.sin(th + w * dt),
        y + v / w * np.cos(th) - v / w * np.cos(th + w * dt),
        wrap(th + w * dt),
    ])


def G(a, s, dt=dt):
    """Jacobian of g with respect to the state."""
    v, w = twist(a)
    th = s[2]
    if abs(w * dt) < STRAIGHT:
        return np.array([[1, 0, -v * dt * np.sin(th)], [0, 1, v * dt * np.cos(th)], [0, 0, 1]])
    return np.array([
        [1, 0, v / w * (np.cos(th + w * dt) - np.cos(th))],
        [0, 1, v / w * (np.sin(th + w * dt) - np.sin(th))],
        [0, 0, 1],
    ])


def V(a, s, dt=dt):
    """Jacobian of g with respect to the wheel speeds (u_L, u_R)."""
    v, w = twist(a)
    th = s[2]
    s0, c0, s1, c1 = np.sin(th), np.cos(th), np.sin(th + w * dt), np.cos(th + w * dt)
    if abs(w * dt) < STRAIGHT:  # straight line: the limit of the arc derivatives
        dg_dvw = np.array([[c0 * dt, -v * s0 * dt**2 / 2], [s0 * dt, v * c0 * dt**2 / 2], [0, dt]])
    else:
        dg_dvw = np.array([
            [(s1 - s0) / w, v * (s0 - s1) / w**2 + v * c1 * dt / w],
            [(c0 - c1) / w, -v * (c0 - c1) / w**2 + v * s1 * dt / w],
            [0, dt],
        ])
    return dg_dvw @ J


def h(s, j, m):
    """Range and bearing from the pose s to landmark j of the map m."""
    dx, dy = m[j] - s[:2]
    return np.array([np.hypot(dx, dy), wrap(np.arctan2(dy, dx) - s[2])])


def H(s, j, m):
    """Jacobian of h with respect to the state."""
    dx, dy = m[j] - s[:2]
    q = dx**2 + dy**2
    sq = np.sqrt(q)
    return np.array([[-sq * dx, -sq * dy, 0], [dy, -dx, -q]]) / q

In [ ]:
GATE = 9.21  # 99% point of the chi-square distribution with 2 degrees of freedom


def ekf_localization(mu, Sigma, a, z, c, m, M, R, dt=dt, gate=None, log=None):
    """One step of EKF localization with known correspondences.

    z holds one (range, bearing) row per observed landmark and c the landmark index of each row.
    With a gate, a feature whose squared Mahalanobis distance exceeds it is rejected.
    If log is a list, each feature appends (squared Mahalanobis distance, accepted).
    """
    G_t, V_t = G(a, mu, dt), V(a, mu, dt)
    mu_bar = g(a, mu, dt)  # line 1
    Sigma_bar = G_t @ Sigma @ G_t.T + V_t @ M @ V_t.T  # line 2
    for z_i, j in zip(z, c):  # line 3
        z_hat, H_i = h(mu_bar, j, m), H(mu_bar, j, m)  # line 4
        S_i = H_i @ Sigma_bar @ H_i.T + R  # line 5
        nu = z_i - z_hat  # line 6
        nu[1] = wrap(nu[1])
        d2 = nu @ np.linalg.solve(S_i, nu)  # line 7
        accepted = gate is None or d2 <= gate
        if log is not None:
            log.append((d2, accepted))
        if not accepted:
            continue
        K_i = Sigma_bar @ H_i.T @ np.linalg.inv(S_i)  # line 8
        mu_bar = mu_bar + K_i @ nu  # line 9
        mu_bar[2] = wrap(mu_bar[2])
        I_KH = np.eye(3) - K_i @ H_i
        Sigma_bar = I_KH @ Sigma_bar @ I_KH.T + K_i @ R @ K_i.T  # line 10
    return mu_bar, Sigma_bar  # line 11

### One step with one landmark

Start with a single step, the setting of the [EKF section](/aiml-common/lectures/rse/ekf/index). The robot drives straight down the page, at heading $\theta = -90°$, at 0.25 m/s for 4 s. Its belief knows the starting position to 2 cm but the heading only to $30°$, and the motion noise is left out to isolate the heading. A single landmark, $m_1 = (0, -2)$ m, stands straight ahead. The robot's true heading during the drive was $25°$ off the nominal one, so it really stopped at about $(0.42, -0.91)$ m. From there its sensor measures the range and bearing to $m_1$, with noise $\sigma_r = 5$ cm and $\sigma_\phi = 2°$.

The code runs one step of `ekf_localization`. For comparison it also approximates the true posterior by Monte Carlo. It pushes 100,000 samples of the starting pose through $g$ and weights each one by the likelihood of the measurement. The weighted samples give the posterior mean and standard deviation, and their *effective sample size* says how many unweighted samples they are worth.

In [ ]:
sigma_r, sigma_phi = 0.05, np.deg2rad(2)  # range and bearing noise
R = np.diag([sigma_r**2, sigma_phi**2])

u = 0.25 / r
a_straight = np.array([u, u])  # equal wheel speeds: straight ahead at 0.25 m/s
m_one = np.array([[0.0, -2.0]])
mu0 = np.array([0.0, 0.0, -np.pi / 2])
Sigma0 = np.diag([0.02**2, 0.02**2, np.deg2rad(30) ** 2])
no_noise = np.zeros((2, 2))

# the true pose and the measurement taken from it
s_true1 = g(a_straight, mu0 + np.array([0, 0, np.deg2rad(25)]), dt=4.0)
z1 = h(s_true1, 0, m_one) + np.random.normal(0, [sigma_r, sigma_phi])

# EKF: prediction alone, then the full step with the measurement (no gate)
mu_pred, Sigma_pred = ekf_localization(mu0, Sigma0, a_straight, [], [], m_one, no_noise, R, dt=4.0)
log1 = []
mu1, Sigma1 = ekf_localization(mu0, Sigma0, a_straight, [z1], [0], m_one, no_noise, R, dt=4.0, log=log1)

# Monte Carlo posterior: push samples through g and weight them by the measurement likelihood
n_samples = 100_000
pred = g(a_straight, np.random.multivariate_normal(mu0, Sigma0, n_samples).T, dt=4.0)
dx, dy = m_one[0, 0] - pred[0], m_one[0, 1] - pred[1]
res_r = z1[0] - np.hypot(dx, dy)
res_phi = wrap(z1[1] - wrap(np.arctan2(dy, dx) - pred[2]))
weights = np.exp(-(res_r**2) / (2 * sigma_r**2) - res_phi**2 / (2 * sigma_phi**2))
weights /= weights.sum()
post_mean = pred[:2] @ weights
post_std = np.sqrt((pred[:2] - post_mean[:, None]) ** 2 @ weights)
ess = 1 / np.sum(weights**2)  # effective sample size

print("measured range, bearing:", z1.round(3), " predicted:", h(mu_pred, 0, m_one).round(3))
print("true          x, y:", s_true1[:2].round(3))
print("EKF predicted x, y:", mu_pred[:2].round(3), "std:", np.sqrt(np.diag(Sigma_pred)[:2]).round(3))
print("EKF updated   x, y:", mu1[:2].round(3), "std:", np.sqrt(np.diag(Sigma1)[:2]).round(3))
print("Monte Carlo   x, y:", post_mean.round(3), "std:", post_std.round(3), f" effective sample size {ess:.0f}")
print(f"squared Mahalanobis distance of the measurement: {log1[0][0]:.1f}")

In [ ]:
def ellipse(mean, cov, k=2.0, n=100):
    """Points on the k-sigma ellipse of a 2D Gaussian."""
    vals, vecs = np.linalg.eigh(cov)
    t = np.linspace(0, 2 * np.pi, n)
    return mean[:, None] + vecs @ (k * np.sqrt(vals)[:, None] * np.vstack([np.cos(t), np.sin(t)]))


import matplotlib.patheffects as pe

halo = [pe.withStroke(linewidth=4, foreground="white")]
post_draws = pred[:, np.random.choice(n_samples, size=3000, p=weights)]  # unweighted draws, for drawing only

fig, (ax_m, ax_p) = plt.subplots(1, 2, figsize=(6.5, 4.2), sharey=True)
for ax in (ax_m, ax_p):
    ax.plot(*m_one[0], "*", color=GOLD, ms=12, mec="black", mew=0.5, label="landmark $m_1$")
    ax.plot(*s_true1[:2], "s", color=GREEN, ms=6, mec="white", label="true position", zorder=6)
    ax.set_aspect("equal")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-2.2, 0.9)
    ax.set_xlabel("$x$ (m)")

ax_m.scatter(pred[0, :3000], pred[1, :3000], s=2, color=VERMILLION, alpha=0.25, rasterized=True,
             label="predicted samples")
e = ellipse(mu_pred[:2], Sigma_pred[:2, :2])
ax_m.plot(e[0], e[1], color=PINK, lw=2, path_effects=halo, zorder=5, label=r"EKF $2\sigma$")
ax_m.set_ylabel("$y$ (m)")
ax_m.set_title("prediction")
ax_m.legend(loc="upper center", fontsize=7)

ax_p.scatter(post_draws[0], post_draws[1], s=2, color=BLUE, alpha=0.3, rasterized=True,
             label="Monte Carlo posterior")
e = ellipse(mu1[:2], Sigma1[:2, :2])
ax_p.plot(e[0], e[1], color=PINK, lw=2, path_effects=halo, zorder=5, label=r"EKF $2\sigma$")
ax_p.plot(*mu1[:2], "o", color=PINK, ms=6, mec="white", label="EKF mean", zorder=7)
ax_p.set_title("after the update")
ax_p.legend(loc="upper center", fontsize=7)
fig.tight_layout()
plt.show()

*Left: samples of the predicted pose (orange) and the EKF's predicted $2\sigma$ ellipse. Right: draws from the Monte Carlo posterior (blue) and the EKF's updated mean with its small $2\sigma$ ellipse. The star is the landmark and the green square the true position.*

The predicted samples form the banana from the [overconfident EKF prediction](/aiml-common/lectures/rse/ekf-overconfident-prediction/index) section, opening upward because the robot drives down the page. The EKF's prediction ellipse is wide in $x$, where a heading error moves the robot sideways. In $y$, the direction of travel, its standard deviation is only 2 cm, because the Jacobian passes no heading uncertainty into the forward position.

The two parts of the measurement do different jobs. The bearing, $-0.82$ rad or about $-47°$, puts the landmark well to the robot's right. Only samples on the right arm of the banana agree with that, so the bearing removes the left-right ambiguity that a range alone would leave. The EKF uses it too. At the predicted mean the bearing row of $H_t$ couples $x$ and $\theta$, and the update moves the mean to $x = 0.41$ m, close to the true 0.42 m, with 2 cm of uncertainty.

The range, 1.257 m against a predicted 1.0 m, says the robot stopped short. Here the EKF falls short. Its predicted $y$ uncertainty was only 2 cm, so it moves $y$ only from $-1.0$ to $-0.965$ m, and the true $y = -0.906$ m ends up about three of its standard deviations away. The Monte Carlo posterior, centered at $(0.44, -0.89)$ m, has no such problem. Its weights are uneven, because few samples agree with the measurement, so the 100,000 samples are worth about 3,100 unweighted ones. With a standard deviation of 2 cm, that still puts the Monte Carlo error of its mean below a millimeter. This is the overconfidence from the EKF section, now inside a localization step.

The overconfidence also shows in the innovation. Under the EKF's prediction the measurement has a squared Mahalanobis distance of 23.4, far beyond the gate $\gamma = 9.21$ of line 7. This example runs without the gate. With it, the EKF would have rejected the one measurement that could correct it.

### Driving among landmarks

Over many steps, the code below drives a simulated TurtleBot3 Waffle around a circle of radius 1.5 m at 0.2 m/s for 70 s, about one and a half laps. Three landmarks stand outside the circle. The sensor is idealized: it sees every landmark within 1.5 m, in any direction, with nothing in the way. A camera has a limited field of view and landmarks can hide behind each other, which would leave more steps without a measurement. Even so, on parts of the circle no landmark is within range.

Each wheel of the simulated robot turns at its commanded speed plus noise, with a standard deviation of 5% of the speed plus 0.1 rad/s. The filter knows this noise model, which gives it $M_t$. The sensor noise is the same as in the single step. The filter starts from the nominal starting pose, and the true starting pose is drawn from the filter's initial belief, so the initial error matches the initial covariance. For comparison, the same filter also runs without any measurements, which is dead reckoning from the wheel speeds alone. The gate is off.

In [ ]:
# map: three landmarks outside the circle
m = 2.5 * np.array([[np.cos(t), np.sin(t)] for t in np.deg2rad([0, 120, 240])])
max_range = 1.5

# commanded wheel speeds for a counterclockwise circle of radius 1.5 m at 0.2 m/s
v_cmd, w_cmd = 0.2, 0.2 / 1.5
a_cmd = np.array([(v_cmd - d * w_cmd) / r, (v_cmd + d * w_cmd) / r])
sigma_u = 0.05 * np.abs(a_cmd) + 0.1
M = np.diag(sigma_u**2)

n_steps = 140
mu_start = np.array([1.5, 0.0, np.pi / 2])
Sigma_start = np.diag([0.05**2, 0.05**2, np.deg2rad(5) ** 2])


def in_view(s):
    """Indices of the landmarks within max_range of the pose s, in any direction."""
    return [j for j in range(len(m)) if h(s, j, m)[0] <= max_range]


def drive(seed, gate=None):
    """Drive the circle once with the EKF and with dead reckoning; return the histories."""
    rng = np.random.default_rng(seed)
    s = rng.multivariate_normal(mu_start, Sigma_start)  # true start, drawn from the initial belief
    mu, Sigma = mu_start.copy(), Sigma_start.copy()
    mu_dr, Sigma_dr = mu.copy(), Sigma.copy()
    hist = {"s": [s], "mu": [mu], "Sigma": [Sigma], "mu_dr": [mu_dr], "Sigma_dr": [Sigma_dr],
            "n_seen": [len(in_view(s))], "log": []}
    for t in range(n_steps):
        s = g(a_cmd + rng.normal(0, sigma_u), s)  # the robot moves with noisy wheels
        c = in_view(s)
        z = [h(s, j, m) + rng.normal(0, [sigma_r, sigma_phi]) for j in c]
        mu, Sigma = ekf_localization(mu, Sigma, a_cmd, z, c, m, M, R, gate=gate, log=hist["log"])
        mu_dr, Sigma_dr = ekf_localization(mu_dr, Sigma_dr, a_cmd, [], [], m, M, R)
        for key, val in zip(("s", "mu", "Sigma", "mu_dr", "Sigma_dr", "n_seen"),
                            (s, mu, Sigma, mu_dr, Sigma_dr, len(c))):
            hist[key].append(val)
    return {key: np.array(val) for key, val in hist.items()}


run = drive(seed=7)
err = run["mu"] - run["s"]  # estimate minus truth
err[:, 2] = wrap(err[:, 2])
inside = np.abs(err) <= 2 * np.sqrt(run["Sigma"][:, [0, 1, 2], [0, 1, 2]])
print(f"steps with a landmark in view: {np.mean(run['n_seen'] > 0):.0%}")
print("final position error, EKF:", np.round(np.hypot(*err[-1, :2]), 3), "m",
      " dead reckoning:", np.round(np.hypot(*(run["mu_dr"][-1, :2] - run["s"][-1, :2])), 3), "m")
print("fraction of steps inside 2 sigma (x, y, theta):", inside.mean(axis=0).round(2))

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(6.5, 6.9))
for j, lm in enumerate(m):
    ax.add_patch(plt.Circle(lm, max_range, color=GOLD, alpha=0.1))
    ax.plot(*lm, "*", color=GOLD, ms=14, mec="black", mew=0.5, label="landmark" if j == 0 else None)
    ax.annotate(f"$m_{j + 1}$", lm, xytext=(6, 6), textcoords="offset points")
ax.plot(run["s"][:, 0], run["s"][:, 1], color=GREEN, lw=1.5, label="true path")
ax.plot(run["mu_dr"][:, 0], run["mu_dr"][:, 1], color=GRAY, ls=":", lw=1.5, label="dead reckoning")
ax.plot(run["mu"][:, 0], run["mu"][:, 1], color=VERMILLION, lw=1, label="EKF mean")
for t in range(0, n_steps + 1, 14):
    seen = run["n_seen"][t] > 0
    e = ellipse(run["mu"][t, :2], run["Sigma"][t, :2, :2])
    ax.plot(e[0], e[1], color=VERMILLION if seen else BLUE, lw=1.4, ls="-" if seen else "--")
ax.plot([], [], color=VERMILLION, lw=1.4, label=r"EKF $2\sigma$, landmark in view")
ax.plot([], [], color=BLUE, lw=1.4, ls="--", label=r"EKF $2\sigma$, none in view")
handles, labels = ax.get_legend_handles_labels()
handles.insert(1, Patch(color=GOLD, alpha=0.25))
labels.insert(1, "sensing range")
ax.set_aspect("equal")
ax.set_xlim(-3.2, 3.2)
ax.set_ylim(-3.0, 3.0)
ax.set_xlabel("$x$ (m)")
ax.set_ylabel("$y$ (m)")
ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
fig.tight_layout()
plt.show()

*Paths of one run. Each ellipse is the EKF's $2\sigma$ position uncertainty, drawn every 7 s: solid where a landmark is in view, dashed where none is. The shaded disks are the sensing ranges.*

In [ ]:
time = np.arange(n_steps + 1) * dt
fig, axes = plt.subplots(3, 1, figsize=(6.5, 5.8), sharex=True)
labels = ["$x$ error (m)", "$y$ error (m)", r"$\theta$ error (rad)"]
for k, (ax, lab) in enumerate(zip(axes, labels)):
    bound = 2 * np.sqrt(run["Sigma"][:, k, k])
    ymax = 1.3 * max(bound.max(), np.abs(err[:, k]).max())
    ax.fill_between(time, -ymax, ymax, where=run["n_seen"] == 0, color=GRAY, alpha=0.25, step="mid", lw=0,
                    label="no landmark in view")
    ax.fill_between(time, -bound, bound, color=VERMILLION, alpha=0.2, lw=0, label=r"EKF $\pm 2\sigma$")
    ax.plot(time, err[:, k], color=VERMILLION, lw=1, label="EKF error")
    ax.set_ylim(-ymax, ymax)
    ax.set_ylabel(lab)
axes[-1].set_xlabel("time (s)")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0))
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

*Errors of the same run, estimate minus truth, with the EKF's $\pm 2\sigma$ band. Gray stretches have no landmark in view.*

Without measurements the estimate drifts: after 70 s dead reckoning is 34 cm from the true position, against 9 cm for the EKF. The dashed ellipses, drawn where no landmark is in view, are larger than the solid ones. The error figure shows the same thing over time: the band widens through each gray stretch and narrows within a step or two of a landmark coming back into range.

The error stays inside the band at 98 to 100% of the steps of this run. That does not show that the filter is consistent. The steps of one run are strongly correlated, since an error at one step is still there at the next, and a band per coordinate does not test the joint covariance of $x$, $y$ and $\theta$.

### Checking consistency over many runs

A filter is *consistent* when its error has the covariance the filter claims. The *normalized estimation error squared* (NEES) tests this for the whole state at once:

$$
\epsilon_t = (\mu_t - s_t)^\top \Sigma_t^{-1} (\mu_t - s_t).
$$

For a consistent filter $\epsilon_t$ follows a chi-square distribution with 3 degrees of freedom, the size of the state, so its mean is 3. The NEES needs the true state, which only a simulation has. The *normalized innovation squared* (NIS) needs only the measurements, so a real robot can compute it. It is the squared Mahalanobis distance $\nu^\top S^{-1} \nu$ of line 7, with mean 2 for a consistent filter.

The code below repeats the drive 50 times with different noise and averages the NEES over the runs at each time step. A consistent filter's average falls inside a band given by the chi-square distribution 95% of the time. The code does this once with the gate off and once with it on.

In [ ]:
from scipy.stats import chi2

n_runs = 50


def consistency(gate):
    """Average NEES at each time step over n_runs runs, all NIS values, and the rejections per run."""
    nees, nis, rejected = np.zeros((n_runs, n_steps + 1)), [], []
    for k in range(n_runs):
        run_k = drive(seed=100 + k, gate=gate)
        e = run_k["mu"] - run_k["s"]
        e[:, 2] = wrap(e[:, 2])
        nees[k] = np.einsum("ti,tij,tj->t", e, np.linalg.inv(run_k["Sigma"]), e)
        nis.extend(run_k["log"][:, 0])
        rejected.append(np.sum(run_k["log"][:, 1] == 0))
    return nees.mean(axis=0), np.array(nis), np.array(rejected)


lo, hi = chi2.ppf([0.025, 0.975], df=3 * n_runs) / n_runs  # 95% band for the average of n_runs NEES values
for gate in (None, GATE):
    avg_nees, nis, rejected = consistency(gate)
    if gate is None:
        avg_nees_free = avg_nees
    else:
        avg_nees_gated = avg_nees
    print(f"gate {gate}: average NEES {avg_nees.mean():.2f} (3 if consistent), "
          f"{np.mean((avg_nees >= lo) & (avg_nees <= hi)):.0%} of time steps inside [{lo:.2f}, {hi:.2f}], "
          f"average NIS {nis.mean():.2f} (2 if consistent)")
    if gate is not None:
        print(f"  rejected features: {rejected.sum()} of {len(nis)}, in {np.sum(rejected > 0)} runs; "
              f"runs with 10 or more rejections: {np.sum(rejected >= 10)}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.axhspan(lo, hi, color=GREEN, alpha=0.15, lw=0, label="95% band for a consistent filter")
ax.axhline(3, color=GREEN, lw=1, ls="--")
ax.plot(time, avg_nees_free, color=VERMILLION, lw=1.2, label="no gate")
ax.plot(time, avg_nees_gated, color=BLUE, lw=1.2, ls="--", label="with the gate")
ax.set_xlabel("time (s)")
ax.set_ylabel(f"average NEES, {n_runs} runs")
ax.set_ylim(0, 1.1 * max(avg_nees_free.max(), avg_nees_gated.max()))
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

*Average NEES over 50 runs at each time step, with the gate off (solid) and on (dashed). The shaded band is where the average of a consistent filter falls 95% of the time.*

Without the gate the average NIS is 2.05, close to 2, so the innovations are as large as the filter predicts. The average NEES is 3.42 rather than 3. It is close to the band for the first 40 s and mostly above it afterwards, and it is inside the band at 77% of the time steps, so the filter is mildly overconfident in the second lap. This section does not isolate the cause. A likely one is the linearization, since the heading uncertainty grows in each stretch with no landmark in view.

With the gate the filter does worse, although the simulation produces no bad features. The gate rejected 127 of 3,590 features, in 27 of the 50 runs. If the filter were consistent it would reject about 1% of them, 36. In 4 runs it rejected 10 or more. Once the estimate has drifted, correct features fall outside the gate, the filter rejects the measurements that would pull it back, and the error grows further. The average NIS rises to 4.80 and the average NEES to 3.98. A gate guards against bad features, but a filter that uses one also needs to notice when it keeps rejecting good ones, for example by counting rejections and relocalizing with a [particle filter](/aiml-common/lectures/rse/particle-filter/index).

## Where road maps come from

Lane-level road maps have been built by surveying. The markings are measured with surveying instruments such as a total station, which is slow for the many markings of a large area ([arXiv:2209.07737](https://arxiv.org/abs/2209.07737)), or taken from the point clouds of vehicles carrying 3D LIDAR, which is costly ([arXiv:2601.19536](https://arxiv.org/abs/2601.19536)). Camera-only pipelines are cheaper. Each frame is warped to a [bird's-eye view](/aiml-common/lectures/planning/local-planning/birds-eye-view/index) with the homography of the road plane, and the lane markings are segmented there. The marking positions and the homography are then refined so that markings seen in many frames land in the same place on the road. The first paper takes the vehicle poses from RTK-GNSS and refines the markings and the homography. The second refines the vehicle poses as well.

Mobileye's REM collects lane marks and landmarks from the cameras of production cars and merges them in the cloud into a map that cars localize against by detecting its landmarks ([Mobileye](https://www.mobileye.com/technology/rem/)). Such a map holds much more than the point landmarks of this section. The localization step follows the same idea: the car predicts where the map's landmarks should appear and corrects its pose from where it sees them.